In [4]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [ ]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()
DB = "airline_prices.db"

OpenAI API Key exists and begins sk-proj-


In [16]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
If the user wants to make a booking, call the book_ticket tool.
Always be accurate. If you don't know the answer, say so.
"""

In [21]:
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS bookings (city TEXT, name TEXT, email TEXT)')
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT, price REAL)')
    conn.commit()

data = [
    ('london', 100),
    ('paris', 200),
    ('new york', 300),
    ('tokyo', 400),
    ('sydney', 500)
]

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.executemany('INSERT INTO prices (city, price) VALUES (?, ?)', data)
    conn.commit()


In [6]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [8]:
def book_ticket(city, name, email):
    print(f"DATABASE TOOL CALLED: Booking ticket for {name} to {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO bookings (city, name, email) VALUES (?, ?, ?)', (city.lower(), name, email))
        conn.commit()
        return f"Ticket booked successfully for {name} to {city}"   

In [ ]:
book_ticket("london", "John Doe", "john.doe@example.com")

In [15]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

booking_function = {
    "name": "book_ticket",
    "description": "Book a ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "name": {
                "type": "string",
                "description": "The name of the customer",
            },
            "email": {
                "type": "string",
                "description": "The email of the customer",
            },
        },
        "required": ["destination_city", "name", "email"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_function}, {"type": "function", "function": booking_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'book_ticket',
   'description': 'Book a ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'},
     'name': {'type': 'string', 'description': 'The name of the customer'},
     'email': {'type': 'string', 'description': 'The email of the customer'}},
    'required': ['destination_city', 'name', 'email'],
    'additionalProperties': False}}}]

In [24]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [23]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)
        
        if tool_call.function.name == "get_ticket_price":
            city = arguments.get('destination_city')
            result = get_ticket_price(city)
            
        elif tool_call.function.name == "book_ticket":
            city = arguments.get('destination_city')
            name = arguments.get('name')
            email = arguments.get('email')
            result = book_ticket(city, name, email)
        
        else:
            result = "Error: Unknown tool"

        responses.append({
            "role": "tool",
            "content": str(result),
            "tool_call_id": tool_call.id
        })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()